# Figure 3B and 3F | Striatal MSN refinement and diffusion-map S–M axis

Hierarchical refinement of striatal MSN subtypes (Figure 3B) and construction of the continuous striosome–matrix (S–M) transcriptional axis using diffusion maps (Figure 3F).

Check input/output paths before running. Run the notebook from top to bottom with a fresh kernel.

| Panel | Workflow |
|---|---|
| B | Hierarchical refinement of striatal MSN populations and final MSN-subtype UMAP |
| F | D1 MSN S/M program scoring, diffusion-map embedding, DC1 visualization, and DC1–program correlations |

The uploaded diffusion-map source notebook contains the complete D1 implementation. The same conceptual analysis was used for D1/D2 S–M organization in the manuscript, but no separate D2 execution block was present in the supplied source notebook; this release therefore reproduces the recorded D1 workflow without inventing D2-specific steps.


## Required inputs

- `str_neuron_refined_clustered.h5ad`: striatal-neuron object containing the initial `refined_cluster` annotation, `batch`, `total_counts`, and a `.raw` expression matrix.
- `integrated_refined_cluster_refined_domain_20250605.h5ad`: integrated BASSFISH object used to transfer `sample`, `condition`, and `refined_domain_3` metadata to the refined striatal-neuron object.

The source workflow prefixes cell IDs with the batch identifier before refinement. Confirm that the input cell IDs have not already been prefixed before running this step.


## Imports and configuration


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import scanpy.external as sce
import matplotlib as mpl
import matplotlib.pyplot as plt
import palantir

from scipy.stats import linregress, pearsonr

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

sc.settings.set_figure_params(
    dpi=80,
    dpi_save=600,
    figsize=(5, 5),
)

# ---------------------------------------------------------------------
# Paths: edit for the local repository/data layout.
# ---------------------------------------------------------------------
DATA_ROOT = Path("data")
OUT_DIR = Path("outputs/Fig3B_F")
FIG_DIR = OUT_DIR / "figures"
H5AD_DIR = OUT_DIR / "h5ad"

STR_NEURON_INPUT = DATA_ROOT / "str_neuron_refined_clustered.h5ad"
INTEGRATED_DOMAIN_INPUT = (
    DATA_ROOT / "integrated_refined_cluster_refined_domain_20250605.h5ad"
)

STR_NEURON_FINAL = H5AD_DIR / "str_neuron_final.h5ad"
D1_SM_FINAL = H5AD_DIR / "d1_sm_data_final.h5ad"

FIG_DIR.mkdir(parents=True, exist_ok=True)
H5AD_DIR.mkdir(parents=True, exist_ok=True)


# Part I | Figure 3B — Hierarchical refinement of striatal MSNs

The original workflow first re-integrated the major striatal neuronal populations with Harmony and separated matrix-, striosome-, NAcc-, ventral-striatal-, and D1-hybrid-related states. Dorsal D1 and D2 populations were then refined separately to distinguish dorsal matrix- and striosome-associated MSN states.


## 1. Load the initial striatal-neuron object


In [ ]:
str_neuron = sc.read_h5ad(STR_NEURON_INPUT)

required_obs = {"batch", "refined_cluster", "total_counts"}
missing_obs = required_obs - set(str_neuron.obs.columns)
if missing_obs:
    raise KeyError(f"Missing required obs columns: {sorted(missing_obs)}")

if str_neuron.raw is None:
    raise ValueError("The input striatal-neuron object must contain `.raw`.")

print(str_neuron)
print(str_neuron.obs["refined_cluster"].value_counts())


In [ ]:
def prefix_cell_ids_with_batch(adata):
    """Reproduce the cell-ID convention used in the source workflow."""
    adata = adata.copy()
    adata.obs_names = (
        adata.obs["batch"].astype(str)
        + "_"
        + adata.obs_names.astype(str)
    )
    return adata


str_neuron = prefix_cell_ids_with_batch(str_neuron)


## 2. Harmony preprocessing helper

The same preprocessing function was reused throughout the refinement workflow: total-count normalization, log transformation, regression of total counts, scaling, PCA, Harmony correction by batch (`theta=1.5`, `max_iter_harmony=10`), a 15-nearest-neighbor graph using 30 Harmony PCs, and UMAP with `min_dist=0.1`.


In [ ]:
def preprocessing_harmony(adata):
    adata = adata.copy()

    print("Input shape:", adata.shape)

    adata.raw = adata
    adata.layers["raw"] = adata.X.copy()

    sc.pp.normalize_total(adata, target_sum=None)
    sc.pp.log1p(adata)
    sc.pp.regress_out(adata, ["total_counts"])
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")

    sc.external.pp.harmony_integrate(
        adata,
        key=["batch"],
        basis="X_pca",
        adjusted_basis="X_pca_harmony",
        lamb=None,
        theta=[1.5],
        max_iter_harmony=10,
    )

    sc.pp.neighbors(
        adata,
        n_neighbors=15,
        n_pcs=30,
        use_rep="X_pca_harmony",
    )
    sc.tl.umap(adata, min_dist=0.1)

    return adata


## 3. First-stage MSN refinement

Cells initially annotated as `STR_Erbb4_neuron`, `MSN_D2`, or `MSN_D1` were re-integrated and clustered at Leiden resolution 0.3. The small initial cluster 4 was removed and the retained cells were reclustered at the same resolution.


In [ ]:
INITIAL_MSN_CLASSES = [
    "STR_Erbb4_neuron",
    "MSN_D2",
    "MSN_D1",
]

str_neuron_sub = str_neuron[
    str_neuron.obs["refined_cluster"].isin(INITIAL_MSN_CLASSES)
].copy()

str_neuron_sub = str_neuron_sub.raw.to_adata()
str_neuron_sub = preprocessing_harmony(str_neuron_sub)

sc.tl.leiden(str_neuron_sub, resolution=0.3)

# Small outlier cluster removed in the source analysis.
str_neuron_sub = str_neuron_sub[
    str_neuron_sub.obs["leiden"] != "4"
].copy()

sc.tl.leiden(str_neuron_sub, resolution=0.3)

str_neuron_sub.obs["refined_cluster_2"] = (
    str_neuron_sub.obs["leiden"].astype(str)
)

print(str_neuron_sub.obs["leiden"].value_counts())


### 3.1 Refinement of first-stage cluster 1

Cluster 1 was reclustered at resolution 0.3. Its internal cluster 1 was further reclustered at resolution 0.4 to separate D2-associated NAcc/ventral states from an unknown population.


In [ ]:
str_neuron_sub2 = str_neuron_sub[
    str_neuron_sub.obs["leiden"] == "1"
].copy()

sc.tl.leiden(str_neuron_sub2, resolution=0.3)

str_neuron_sub3 = str_neuron_sub2[
    str_neuron_sub2.obs["leiden"] == "1"
].copy()

sc.tl.leiden(str_neuron_sub3, resolution=0.4)

str_neuron_sub3.obs["refined_cluster_3"] = (
    str_neuron_sub3.obs["leiden"]
    .astype(str)
    .replace({
        "0": "S_D2",
        "1": "O_D2",
        "2": "Unknown",
    })
)

str_neuron_sub2.obs["refined_cluster_2"] = (
    str_neuron_sub2.obs["leiden"].astype(str)
)

str_neuron_sub2.obs.loc[
    str_neuron_sub3.obs_names,
    "refined_cluster_2",
] = str_neuron_sub3.obs["refined_cluster_3"].astype(str)

str_neuron_sub2.obs["refined_cluster_2"] = (
    str_neuron_sub2.obs["refined_cluster_2"].replace({
        "0": "O_D1",
        "2": "S_D1",
        "3": "In_Foxp2_Olfm3_2",
    })
)

str_neuron_sub.obs.loc[
    str_neuron_sub2.obs_names,
    "refined_cluster_2",
] = str_neuron_sub2.obs["refined_cluster_2"].astype(str)


### 3.2 Refinement of the matrix-associated first-stage cluster

First-stage cluster 0 was reclustered at Leiden resolution 0.2 and separated into D1- and D2-associated matrix populations.


In [ ]:
matrix_neuron = str_neuron_sub[
    str_neuron_sub.obs["leiden"] == "0"
].copy()

sc.tl.leiden(matrix_neuron, resolution=0.2)

matrix_neuron.obs["refined_matrix_cluster"] = (
    matrix_neuron.obs["leiden"]
    .astype(str)
    .replace({
        "0": "M_D2",
        "1": "M_D1",
    })
)

str_neuron_sub.obs.loc[
    matrix_neuron.obs_names,
    "refined_cluster_2",
] = matrix_neuron.obs["refined_matrix_cluster"].astype(str)


In [ ]:
# The source workflow assigned the remaining first-stage cluster 2 to a
# Foxp2/Olfm3-related population and renamed the two related states.
mask_cluster2 = str_neuron_sub.obs["refined_cluster_2"].astype(str) == "2"
str_neuron_sub.obs.loc[
    mask_cluster2, "refined_cluster_2"
] = "In_Foxp2_Olfm3_1"

str_neuron_sub.obs["refined_cluster_2"] = (
    str_neuron_sub.obs["refined_cluster_2"].astype(str).replace({
        "In_Foxp2_Olfm3_1": "M_In_Foxp2_Olfm3",
        "In_Foxp2_Olfm3_2": "S_In_Foxp2_Olfm3",
    })
)

# Remove cells labeled Unknown.
str_neuron_sub = str_neuron_sub[
    str_neuron_sub.obs["refined_cluster_2"] != "Unknown"
].copy()

print(str_neuron_sub.obs["refined_cluster_2"].value_counts())


## 4. Propagate refined labels to the full striatal-neuron object

Cells removed during the refinement of the three initial MSN classes are also removed from the full striatal-neuron object, matching the original workflow.


In [ ]:
initial_refinement_ids = str_neuron[
    str_neuron.obs["refined_cluster"].isin(INITIAL_MSN_CLASSES)
].obs_names

retained_refinement_ids = str_neuron_sub.obs_names
removed_ids = initial_refinement_ids.difference(retained_refinement_ids)

str_neuron = str_neuron[
    ~str_neuron.obs_names.isin(removed_ids)
].copy()

str_neuron.obs["refined_cluster_2"] = (
    str_neuron.obs["refined_cluster"].astype(str)
)

str_neuron.obs.loc[
    str_neuron_sub.obs_names,
    "refined_cluster_2",
] = str_neuron_sub.obs["refined_cluster_2"].astype(str)

print("Removed refinement outliers/unknown cells:", len(removed_ids))


## 5. Refine dorsal D1 MSNs

The `M_D1` population was reprocessed with the same Harmony workflow and reclustered at Leiden resolution 0.3. The two resulting clusters were annotated as `Dorsal_M_D1` and `Dorsal_S_D1`.


In [ ]:
m_d1 = str_neuron[
    str_neuron.obs["refined_cluster_2"] == "M_D1"
].copy()

m_d1 = m_d1.raw.to_adata()
m_d1 = preprocessing_harmony(m_d1)

sc.tl.leiden(m_d1, resolution=0.3)

m_d1.obs["refined_cluster_3"] = (
    m_d1.obs["leiden"]
    .astype(str)
    .replace({
        "0": "Dorsal_M_D1",
        "1": "Dorsal_S_D1",
    })
)

print(m_d1.obs["refined_cluster_3"].value_counts())


## 6. Refine dorsal D2 MSNs

The `M_D2` population was reprocessed after excluding a set of glial, vascular, and off-target genes recorded in the source workflow. Leiden clustering was performed at resolution 0.51. Cluster 2 was annotated as `Dorsal_S_D2`; the remaining clusters were assigned to `Dorsal_M_D2`.


In [ ]:
m_d2 = str_neuron[
    str_neuron.obs["refined_cluster_2"] == "M_D2"
].copy()

m_d2 = m_d2.raw.to_adata()

D2_EXCLUDED_GENES = [
    "Mbp", "Ttr", "Gja1", "Slc1a3", "Ctss", "Csf1r", "Hexb",
    "Slco1a4", "Flt1", "Cldn5", "Slc2a1", "Drd1", "Vtn",
    "Slc1a2", "Gfap", "Lingo2", "Thbs4",
]

m_d2 = m_d2[
    :, ~m_d2.var_names.isin(D2_EXCLUDED_GENES)
].copy()

m_d2 = preprocessing_harmony(m_d2)
sc.tl.leiden(m_d2, resolution=0.51)

m_d2.obs["refined_cluster_3"] = (
    m_d2.obs["leiden"]
    .astype(str)
    .replace({
        "0": "Dorsal_M_D2",
        "1": "Dorsal_M_D2",
        "2": "Dorsal_S_D2",
        "3": "Dorsal_M_D2",
        "4": "Dorsal_M_D2",
    })
)

print(m_d2.obs["refined_cluster_3"].value_counts())


## 7. Final MSN subtype annotation

The refined dorsal D1/D2 labels are propagated back to the full striatal-neuron object. NAcc, ventral-striatal, and D1-hybrid-related labels are then renamed to the final manuscript nomenclature.


In [ ]:
str_neuron.obs["refined_cluster_3"] = (
    str_neuron.obs["refined_cluster_2"].astype(str)
)

str_neuron.obs.loc[
    m_d1.obs_names,
    "refined_cluster_3",
] = m_d1.obs["refined_cluster_3"].astype(str)

str_neuron.obs.loc[
    m_d2.obs_names,
    "refined_cluster_3",
] = m_d2.obs["refined_cluster_3"].astype(str)

str_neuron.obs["refined_cluster_3"] = (
    str_neuron.obs["refined_cluster_3"].replace({
        "M_In_Foxp2_Olfm3": "In_Foxp2_Olfm3_1",
        "S_In_Foxp2_Olfm3": "In_Foxp2_Olfm3_2",
        "O_D1": "Ventral_D1",
        "O_D2": "Ventral_D2",
        "S_D1": "NAc_D1",
        "S_D2": "NAc_D2",
    })
)

str_neuron.obs["refined_cluster_4"] = (
    str_neuron.obs["refined_cluster_3"].replace({
        "In_Foxp2_Olfm3_2": "D1H_Nxph4",
        "In_Foxp2_Olfm3_1": "D1H_Cpne7",
    })
)


In [ ]:
STR_NEURON_COLORS = {
    "Dorsal_M_D1": "#F7EBA7",
    "Dorsal_M_D2": "#a0d4a5",
    "Dorsal_S_D1": "#061584",
    "Dorsal_S_D2": "#B34CFF",
    "NAc_D1": "#FFA900",
    "NAc_D2": "#5cb998",
    "Ventral_D1": "#EF6702",
    "Ventral_D2": "#006837",
    "D1H_Nxph4": "#70F1FC",
    "D1H_Cpne7": "#93C9ED",
    "STR_chat": "#C1A3CE",
    "STR_Rreb1_neuron": "#742981",
}


### Figure 3B | Final MSN-subtype UMAP


In [ ]:
sc.settings.set_figure_params(figsize=(8, 8))

sc.pl.umap(
    str_neuron,
    color="refined_cluster_4",
    palette=STR_NEURON_COLORS,
    size=5,
    frameon=False,
    show=False,
)

plt.savefig(
    FIG_DIR / "Fig3B_striatal_MSN_refined_subtypes_UMAP.pdf",
    bbox_inches="tight",
)
plt.close()


In [ ]:
# Optional marker diagnostic corresponding to the source notebook.
marker_genes = [
    g for g in ["Drd1", "Drd2", "Nxph4", "Tac1", "Cpne7"]
    if g in str_neuron.var_names
]

if marker_genes:
    sc.pl.umap(
        str_neuron,
        color=marker_genes,
        size=20,
        cmap="Reds",
        frameon=False,
        show=False,
    )
    plt.savefig(
        FIG_DIR / "Fig3B_MSN_marker_diagnostic.pdf",
        bbox_inches="tight",
    )
    plt.close()


In [ ]:
str_neuron.write_h5ad(STR_NEURON_FINAL)
print("Saved:", STR_NEURON_FINAL)


# Part II | Figure 3F — Diffusion-map S–M transcriptional axis

For Figure 3F, dorsal D1 MSNs (`Dorsal_M_D1` and `Dorsal_S_D1`) are selected and integrated across batches. S- and M-associated transcriptional programs are scored, and Palantir diffusion maps are calculated from Harmony-corrected PCA coordinates.

The source analysis used the second diffusion eigenvector (`DM_EigenVectors[:, 1]`) as `dc_component1`. Diffusion-component signs are mathematically arbitrary; for reproducible biological orientation, the release workflow orients DC1 so that S-score increases from the matrix-rich end toward the striosome-rich end before rescaling it to 0–100.


## 8. Transfer sample, condition, and spatial-domain metadata


In [ ]:
integrated_domain = sc.read_h5ad(INTEGRATED_DOMAIN_INPUT)

common_ids = str_neuron.obs_names.intersection(integrated_domain.obs_names)
if len(common_ids) == 0:
    raise ValueError(
        "No overlapping cell IDs between the refined striatal-neuron "
        "object and the integrated-domain object."
    )

str_neuron_dm = str_neuron[common_ids].copy()
integrated_domain = integrated_domain[common_ids].copy()

for col in ["sample", "condition", "refined_domain_3"]:
    if col not in integrated_domain.obs.columns:
        raise KeyError(f"Required metadata column '{col}' is missing.")
    str_neuron_dm.obs[col] = integrated_domain.obs[col].values

print("Cells retained for diffusion-map preparation:", str_neuron_dm.n_obs)


## 9. Select and preprocess dorsal D1 S/M MSNs

The source workflow retained D1 cells annotated as `Dorsal_M_D1` or `Dorsal_S_D1`, restricted them to striatal spatial domains, required `total_counts > 100`, restored raw expression, and excluded `Ttr`, `Mbp`, `Slc1a3`, and `Gja1` before dimensionality reduction.


In [ ]:
D1_SM_LABELS = ["Dorsal_M_D1", "Dorsal_S_D1"]

STRIATAL_DOMAINS_FOR_DM = [
    "Dorsal_STR_1",
    "Dorsal_STR_2",
    "NAc",
    "Ventral_STR",
    "Ventral_STR_ICj_outer",
    "Ventral_STR_ICj_inner",
]

d1sm_adata = str_neuron_dm[
    str_neuron_dm.obs["refined_cluster_4"].isin(D1_SM_LABELS)
].copy()

d1sm_adata = d1sm_adata[
    d1sm_adata.obs["refined_domain_3"].isin(STRIATAL_DOMAINS_FOR_DM)
].copy()

d1sm_adata = d1sm_adata[
    d1sm_adata.obs["total_counts"] > 100
].copy()

if d1sm_adata.raw is None:
    raise ValueError("D1 S/M object requires `.raw` expression.")

d1sm_adata = d1sm_adata.raw.to_adata()

genes_to_remove = ["Ttr", "Mbp", "Slc1a3", "Gja1"]
d1sm_adata = d1sm_adata[
    :, ~d1sm_adata.var_names.isin(genes_to_remove)
].copy()

print(d1sm_adata)
print(d1sm_adata.obs["refined_cluster_4"].value_counts())


In [ ]:
# Reproduce the D1 diffusion-map preprocessing recorded in the source notebook.
sc.pp.normalize_total(d1sm_adata, target_sum=None)
palantir.preprocess.log_transform(d1sm_adata)

# Preserve the normalized/log-transformed matrix for downstream gene scoring.
d1sm_adata.raw = d1sm_adata.copy()

sc.pp.regress_out(d1sm_adata, ["total_counts"])
sc.pp.pca(d1sm_adata)

sc.external.pp.harmony_integrate(
    d1sm_adata,
    key=["batch"],
    basis="X_pca",
    adjusted_basis="X_pca_harmony",
    lamb=None,
    theta=[1.5],
    max_iter_harmony=10,
)

sc.pp.neighbors(
    d1sm_adata,
    use_rep="X_pca_harmony",
)

sc.tl.umap(
    d1sm_adata,
    n_components=2,
)


## 10. Calculate S- and M-program scores

The gene sets below are the exact sets used in the supplied Figure 3F source notebook.


In [ ]:
S_PROGRAM_GENES = [
    "Tac1",
    "Tshz1",
    "Sorcs1",
    "Pbx3",
    "Sema5b",
    "Serpine2",
    "Pde1c",
]

M_PROGRAM_GENES = [
    "Id4",
    "Sgk1",
    "Epha4",
    "Sv2b",
    "Nefm",
]

missing_s = [g for g in S_PROGRAM_GENES if g not in d1sm_adata.var_names]
missing_m = [g for g in M_PROGRAM_GENES if g not in d1sm_adata.var_names]

if missing_s or missing_m:
    raise KeyError(
        f"Missing S-program genes: {missing_s}; "
        f"missing M-program genes: {missing_m}"
    )

sc.tl.score_genes(
    d1sm_adata,
    gene_list=S_PROGRAM_GENES,
    score_name="S_score",
    use_raw=False,
)

sc.tl.score_genes(
    d1sm_adata,
    gene_list=M_PROGRAM_GENES,
    score_name="M_score",
    use_raw=False,
)

d1sm_adata.obs["SM_score_diff"] = (
    d1sm_adata.obs["S_score"]
    - d1sm_adata.obs["M_score"]
)


In [ ]:
sc.pl.umap(
    d1sm_adata,
    color=["S_score", "M_score"],
    ncols=2,
    frameon=False,
    cmap="seismic",
    show=False,
)

plt.savefig(
    FIG_DIR / "Fig3F_D1_SM_program_scores_UMAP.pdf",
    bbox_inches="tight",
)
plt.close()


## 11. Diffusion-map analysis

Five diffusion components are calculated from `X_pca_harmony` using Palantir. `DM_EigenVectors[:, 1]` is stored as `dc_component1`, matching the source workflow.


In [ ]:
palantir.utils.run_diffusion_maps(
    d1sm_adata,
    n_components=5,
    pca_key="X_pca_harmony",
)

# Retained because it was part of the original diffusion-map workflow.
_ = palantir.utils.determine_multiscale_space(d1sm_adata)

dc = d1sm_adata.obsm["DM_EigenVectors"]

for i in range(min(5, dc.shape[1])):
    d1sm_adata.obs[f"dc_component{i}"] = dc[:, i]

if "dc_component1" not in d1sm_adata.obs:
    raise ValueError("Diffusion-map output does not contain dc_component1.")


### Orient and rescale DC1

The sign of a diffusion component is arbitrary. To keep the manuscript convention consistent across reruns, DC1 is oriented so that it is positively associated with the S program. The oriented coordinate is then linearly rescaled to 0–100, with the matrix-rich end at lower values and the striosome-rich end at higher values.


In [ ]:
r_s_raw, _ = pearsonr(
    d1sm_adata.obs["dc_component1"].astype(float),
    d1sm_adata.obs["S_score"].astype(float),
)

if r_s_raw < 0:
    d1sm_adata.obs["dc_component1"] *= -1

x = d1sm_adata.obs["dc_component1"].astype(float)

d1sm_adata.obs["dc_component1_scale100"] = (
    (x - x.min())
    / (x.max() - x.min())
    * 100
)

print(
    "Oriented DC1–S-score Pearson r:",
    pearsonr(
        d1sm_adata.obs["dc_component1_scale100"],
        d1sm_adata.obs["S_score"],
    )[0],
)


In [ ]:
sc.pl.umap(
    d1sm_adata,
    color="dc_component1_scale100",
    frameon=False,
    cmap="Spectral_r",
    size=5,
    show=False,
)

plt.savefig(
    FIG_DIR / "Fig3F_D1_DC1_UMAP.pdf",
    bbox_inches="tight",
)
plt.close()


## 12. Correlation of DC1 with S- and M-program scores

Pearson correlation and ordinary least-squares linear regression are calculated between the 0–100-scaled DC1 coordinate and each program score. Scatter points are rasterized in the exported PDF to keep file size manageable.


In [ ]:
def plot_score_regression(
    adata,
    y_col,
    x_col="dc_component1_scale100",
    save_path=None,
):
    df = adata.obs[[x_col, y_col]].copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[y_col] = pd.to_numeric(df[y_col], errors="coerce")
    df = df.dropna()

    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)

    regression = linregress(x, y)
    r, p = pearsonr(x, y)

    x_fit = np.linspace(x.min(), x.max(), 200)
    y_fit = regression.intercept + regression.slope * x_fit

    fig, ax = plt.subplots(figsize=(3.2, 2.8))

    ax.scatter(
        x,
        y,
        s=5,
        alpha=0.30,
        edgecolors="none",
        rasterized=True,
    )

    ax.plot(
        x_fit,
        y_fit,
        color="black",
        linewidth=1.2,
    )

    ax.set_xlabel("S–M diffusion axis (DC1, 0–100)")
    ax.set_ylabel(y_col)

    ax.text(
        0.03,
        0.97,
        f"Pearson r = {r:.3f}\nP = {p:.2e}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=7,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(
            save_path,
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()

    return {
        "score": y_col,
        "r": r,
        "p": p,
        "slope": regression.slope,
        "intercept": regression.intercept,
        "n_cells": len(df),
    }


In [ ]:
result_S = plot_score_regression(
    d1sm_adata,
    y_col="S_score",
    save_path=FIG_DIR / "Fig3F_D1_S_score_vs_DC1.pdf",
)

result_M = plot_score_regression(
    d1sm_adata,
    y_col="M_score",
    save_path=FIG_DIR / "Fig3F_D1_M_score_vs_DC1.pdf",
)

correlation_summary = pd.DataFrame([result_S, result_M])
correlation_summary.to_csv(
    OUT_DIR / "Fig3F_D1_DC1_program_correlations.csv",
    index=False,
)

correlation_summary


## 13. Save the D1 S–M diffusion-map object

The saved object contains the S/M program scores, diffusion components, and the oriented 0–100 DC1 coordinate used by downstream analyses.


In [ ]:
d1sm_adata.write_h5ad(D1_SM_FINAL)

print("Saved:", D1_SM_FINAL)
print(
    d1sm_adata.obs[
        [
            "refined_cluster_4",
            "sample",
            "condition",
            "S_score",
            "M_score",
            "dc_component1",
            "dc_component1_scale100",
        ]
    ].head()
)
